In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy comet-ml logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.7 MB/s eta 0:00:00


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [ ]:
import comet_ml
from diploma_code import *

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_cfg = ModelConfig(
    encoder_name="xlm-roberta-base",
    embedding_dim=128,
    query_max_len=256,    # увеличиваем для Phase 2
    doc_max_len=256,
)
model, tokenizer = build_model_and_tokenizer(model_cfg)

ckpt = torch.load("/kaggle/input/models/vikakuznetzowa/checkpoints-phase1/pytorch/default/1/checkpoints/checkpoint_phase1_best.pt",
                   map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel: {n_gpus}x {torch.cuda.get_device_name(0)}")
else:
    print(f"Single GPU: {torch.cuda.get_device_name(0)}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DataParallel: 2x Tesla T4


In [ ]:
import json
import random
with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/hard_negatives.json") as f:
    hn_raw = json.load(f)

random.seed(42)
random.shuffle(hn_raw)
MAX_EXAMPLES = 300_000
hn_raw = hn_raw[:MAX_EXAMPLES]


examples = [
    RetrievalExample(query=r["query"], positive=r["positive"],
                     negatives=r["negatives"], lang=r["lang"])
    for r in hn_raw
]
print(f"Loaded {len(examples)} examples with hard negatives")

Loaded 300000 examples with hard negatives


In [ ]:
data_cfg = DataConfig(mixed_lang_ratio=0.0, cache_dir="/kaggle/temp/cache")

train_loader = build_dataloader(
    examples, tokenizer, model_cfg, data_cfg,
    train_cfg_batch_size=48,
    num_workers=2,
    is_train=True,
)


In [12]:
val_examples = _load_miracl_examples(data_cfg, split="dev")
val_loader = build_dataloader(
    val_examples, tokenizer, model_cfg, data_cfg,
    train_cfg_batch_size=48, num_workers=0, is_train=False,
)

print(f"Train: {len(examples)} examples, {len(train_loader)} batches/epoch (batch_size=48)")
print(f"Val:   {len(val_examples)} examples, {len(val_loader)} batches")

topics.miracl-v1.0-ru-dev.tsv: 0.00B [00:00, ?B/s]

qrels.miracl-v1.0-ru-dev.tsv: 0.00B [00:00, ?B/s]

miracl-corpus-v1.0-ru/docs-0.jsonl.gz:   0%|          | 0.00/100M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-1.jsonl.gz:   0%|          | 0.00/98.0M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-2.jsonl.gz:   0%|          | 0.00/90.1M [00:00<?, ?B/s]

...

miracl-corpus-v1.0-ru/docs-16.jsonl.gz:   0%|          | 0.00/81.4M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-17.jsonl.gz:   0%|          | 0.00/82.0M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-18.jsonl.gz:   0%|          | 0.00/81.0M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-19.jsonl.gz:   0%|          | 0.00/6.75M [00:00<?, ?B/s]

Train: 300000 examples, 6250 batches/epoch (batch_size=48)
Val:   1252 examples, 27 batches


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler

train_cfg = TrainConfig(
    output_dir="/kaggle/working/checkpoints",
    per_device_batch_size=48,
    gradient_accumulation_steps=1,
    learning_rate=1.5e-6,
    fp16=True,
    eval_every_n_steps=500,
    save_every_n_steps=0,
    log_every_n_steps=100,
    max_grad_norm=1.0,
)

num_epochs = 1
total_steps = (len(train_loader) // train_cfg.gradient_accumulation_steps) * num_epochs
print(f"OneCycleLR total_steps = {total_steps}  ({total_steps // num_epochs}/epoch, ~{total_steps * 1.5 / 3600:.1f}h estimated)")

optimizer = AdamW(model.parameters(), lr=train_cfg.learning_rate, weight_decay=0.01)
scheduler = OneCycleLR(optimizer, max_lr=train_cfg.learning_rate,
                       total_steps=max(total_steps, 1), pct_start=0.1)
loss_fn = InfoNCELoss(temperature=0.05)
scaler = GradScaler()

OneCycleLR total_steps = 6250  (6250/epoch, ~2.6h estimated)


/tmp/ipykernel_24/4250802421.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [14]:
comet_cfg = CometConfig(
    enabled=True,
    project_name="colbert-ru",
    experiment_name="phase2-v2-2xT4-fixed-lr",
    tags=["phase2", "hard-negatives", "xlm-roberta-base", "2xT4", "v2"],
)
experiment = _init_comet_experiment(comet_cfg)
if experiment is not None:
    experiment.log_parameters({
        "phase": "phase2",
        "learning_rate": train_cfg.learning_rate,
        "num_epochs": num_epochs,
        "batch_size": train_cfg.per_device_batch_size,
        "grad_accum": train_cfg.gradient_accumulation_steps,
        "num_train_examples": len(examples),
    })

train_one_phase(
    model, train_loader, val_loader,
    optimizer, scheduler, loss_fn, device,
    train_cfg, num_epochs,
    phase_name="phase2", scaler=scaler,
    comet_experiment=experiment,
)

if experiment is not None:
    experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kuznetzowa-viktoria/colbert-ru/a1c1a2da886e4220b1aa7da75cd9afeb

COMET INFO: Couldn't find a Git repository in '/kaggle/working' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
/tmp/ipykernel_24/1379938664.py:207: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : phase2-v2-2xT4-fixe